# Privacy Audit (membership inference + gradient inversion) — Kaggle, run headless

Standalone notebook for the privacy audit in an **over-fitting regime** so the
membership-inference attack actually has signal: a small ~2,000-image training set
trained for 25 epochs, so the undefended model memorizes its data and the attack can
detect it — then differential privacy should remove that leakage. Also runs the
controlled gradient-inversion reconstruction.

## How to run (Save & Run All)
1. Right sidebar: **Accelerator = GPU**, **Internet = On**.
2. **+ Add Input** -> `nih-chest-xrays/data` (label CSVs + images). Optionally also the
   224px mirror `khanfashee/nih-chest-x-ray-14-224x224-resized` for faster image loading.
3. **Save Version -> Save & Run All (Commit)**. It runs headless (~30-45 min); when done,
   the results are in the version's Output (`/kaggle/working/results/audit`), downloadable,
   and `audit.zip` is produced for a one-click download.

No interactive steps (no file upload) — safe for a committed run.


## 1. GPU + install + clone

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/kaggle/working/fl"):
    !git clone -q $REPO_URL /kaggle/working/fl
%cd /kaggle/working/fl
!git log --oneline -1

## 2. Locate the mounted dataset (no download)

In [ ]:
import glob, os, shutil
import pandas as pd
NIH = "/kaggle/working/nih"; os.makedirs(NIH, exist_ok=True)
print("Contents of /kaggle/input:")
for d in sorted(glob.glob("/kaggle/input/*")):
    print("  ", d, sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True)), "png")
all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
labels = next((c for c in all_csv if os.path.basename(c).lower() == "data_entry_2017.csv"), None)
if labels is None:
    for c in all_csv:
        try:
            if {"Image Index","Finding Labels"}.issubset(pd.read_csv(c, nrows=1).columns): labels=c; break
        except Exception: pass
assert labels, "No labels CSV. '+ Add Input' -> 'nih-chest-xrays/data'."
shutil.copy(labels, f"{NIH}/Data_Entry_2017.csv")
bb = next((c for c in all_csv if os.path.basename(c).lower()=="bbox_list_2017.csv"), None)
if bb: shutil.copy(bb, f"{NIH}/BBox_List_2017.csv")
best, best_n = None, 0
for d in sorted(glob.glob("/kaggle/input/*")):
    n = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    if n > best_n: best, best_n = d, n
IMAGE_ROOT = best
print("IMAGE_ROOT:", IMAGE_ROOT, "| png:", best_n); assert best_n > 0
import sys; sys.path.insert(0, "/kaggle/working/fl")

## 3. Configure the over-fitting audit

Small training set + many epochs so the no-DP model memorizes (membership leakage), then
DP (σ=1.0, 2.0) should pull the attack back toward chance (0.5). The audit's loss is NOT
class-balanced, which also helps memorization.

In [ ]:
from src.experiments.config import load_config
cfg = load_config("configs/kaggle_defences.yaml")
cfg["data"]["root"] = NIH
cfg["data"]["image_root"] = IMAGE_ROOT
cfg["partition"]["manifest_path"] = f"{NIH}/manifest_audit.json"
cfg["data"]["subsample"]["max_images"] = 2000     # small -> memorization -> membership leaks
cfg["training"]["local_epochs"] = 25              # over-fit the undefended model
print("audit subset:", cfg["data"]["subsample"]["max_images"], "| epochs:", cfg["training"]["local_epochs"],
      "| image_size:", cfg["data"]["image_size"])

## 4. Run the privacy audit

Trains the no-DP model and DP models (σ=1.0, 2.0), runs the membership-inference attack on
each (ROC-AUC; 0.5 = no leakage), and runs the controlled small-CNN gradient-inversion
reconstruction (SSIM/PSNR with vs without DP).

In [ ]:
from src.experiments.audit import run_privacy_audit
from src.evaluation.audit_report import generate_audit_report
OUT = "/kaggle/working/results/audit"
run_privacy_audit(cfg, OUT, data_root=NIH, dp_noise_levels=(1.0, 2.0),
                  inversion_steps=300, inversion_size=32)
generate_audit_report(OUT)

## 5. Show the results

In [ ]:
from IPython.display import Image, display, Markdown
display(Image(f"{OUT}/plots/09_membership_inference.png"))
display(Image(f"{OUT}/plots/10_gradient_inversion.png"))
display(Markdown(open(f"{OUT}/AUDIT_REPORT.md").read()))

## 6. Download (zip)

In [ ]:
import shutil
from IPython.display import FileLink, display
shutil.make_archive("/kaggle/working/audit", "zip", "/kaggle/working/results/audit")
print("zipped -> /kaggle/working/audit.zip (also saved as the committed version's Output)")
display(FileLink("audit.zip"))